# Predictive maintenance - medallion pipeline

End-to-end exploration organised **by medallion layer**, with **numbered, foldable**
headings: `## phase` > `### source` > `#### sub-section` > `##### feature`. Phases:
**1. Bronze** (raw) > **2. Processing Bronze -> Silver** > **3. Silver** (treated) >
**4. Database** > **5. Cross-source**. Reproducible run: `uv run python scripts/run_pipeline.py`.

## 0. Setup

In [ ]:
import sys, tempfile
from pathlib import Path

%load_ext autoreload
%autoreload 2

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

%matplotlib inline

from src import config
from src.common.env import load_dotenv
from src.common.overview import global_overview
from src.common.processing_summary import processing_summary_markdown
from src.common.profiling import (
    feature_status_badge,
    feature_synthesis_markdown,
    plot_category_counts,
    plot_crosstab_heatmap,
    plot_cumulative_by_machine,
    plot_feature_by_machine,
    plot_feature_distribution,
    plot_keyword_breakdown,
    plot_timeseries_by_machine,
    plot_value_by_machine,
)
from src.sources.registry import SOURCE_SPECS

load_dotenv(PROJECT_ROOT / '.env')

# Source registry drives every phase; data is shared across phases via these dicts.
SPECS = {s.name: s for s in SOURCE_SPECS}
bronze, silver = {}, {}


def show_per_feature(df, numeric_features, count_features=(), count_label='records',
                     keyword_bars=(), heatmaps=(), timeseries=(), bars_by_machine=(),
                     cumulative=(), machine_col=config.MACHINE_COLUMN,
                     feature_plots=None, source=None):
    feature_plots = feature_plots or {}
    out = Path(tempfile.mkdtemp(prefix='nb_'))
    kw_by_feature = {}
    for m, (f, kw, title) in enumerate([t for t in keyword_bars if t[0] in df.columns], start=1):
        kw_by_feature.setdefault(f, []).append((kw, title, m))
    heat_by_row = {}
    present = [(r, c) for r, c in heatmaps if r in df.columns and c in df.columns]
    for k, (r, c) in enumerate(present, start=1):
        heat_by_row.setdefault(r, []).append((c, k))
    ts_by_feature = {}
    present_ts = [(v, t, ti, fr) for v, t, ti, fr in timeseries if v in df.columns and t in df.columns]
    for n, (v, t, ti, fr) in enumerate(present_ts, start=1):
        ts_by_feature.setdefault(v, []).append((t, ti, fr, n))
    bar_by_feature = {}
    for q, v in enumerate([b for b in bars_by_machine if b in df.columns], start=1):
        bar_by_feature.setdefault(v, []).append(q)
    cum_by_feature = {}
    for r, (v, t, ti) in enumerate([c for c in cumulative if c[0] in df.columns and c[1] in df.columns], start=1):
        cum_by_feature.setdefault(v, []).append((t, ti, r))
    for col in df.columns:
        # Feature headings are level 5 so they fold under the "... - Per feature" heading.
        display(Markdown(f'##### {col}{feature_status_badge(df, col, source)}'))
        display(Markdown(feature_synthesis_markdown(df, col, source)))
        if col in numeric_features:
            i = numeric_features.index(col) + 1
            if machine_col in df.columns:
                display(Image(filename=str(plot_feature_by_machine(df, col, i, out, machine_col))))
            display(Image(filename=str(plot_feature_distribution(df, col, i, out))))
        if col in count_features:
            j = list(count_features).index(col) + 1
            display(Image(filename=str(plot_category_counts(df, col, j, out, count_label))))
        for kw, title, m in kw_by_feature.get(col, []):
            display(Image(filename=str(plot_keyword_breakdown(df, col, kw, m, out, title, count_label))))
        for c, k in heat_by_row.get(col, []):
            display(Image(filename=str(plot_crosstab_heatmap(df, col, c, k, out))))
        for t, title, fr, n in ts_by_feature.get(col, []):
            display(Image(filename=str(plot_timeseries_by_machine(df, col, t, machine_col, n, out, freq=fr, title=title))))
        for q in bar_by_feature.get(col, []):
            display(Image(filename=str(plot_value_by_machine(df, col, q, out))))
        for t, title, r in cum_by_feature.get(col, []):
            display(Image(filename=str(plot_cumulative_by_machine(df, col, t, machine_col, r, out, title))))
        plot_fn = feature_plots.get(col)
        if plot_fn is not None:
            for png in plot_fn(df, out):
                display(Image(filename=str(png)))


def show_per_feature_spec(spec, df, numeric_features):
    """Render per-feature understanding using a source spec's declared hooks."""
    show_per_feature(df, numeric_features, spec.count_features, spec.count_label,
                     spec.keyword_bars, spec.heatmaps, spec.timeseries, spec.bars_by_machine,
                     spec.cumulative, spec.machine_col, feature_plots=spec.feature_plots,
                     source=spec.name)


def show_overview(spec, df):
    """Render a source's whole-source overview plots (heading is the markdown cell above)."""
    if spec.overview is None:
        display(Markdown(f'_Overview {spec.name}: a venir._'))
        return
    out = Path(tempfile.mkdtemp(prefix='nb_overview_'))
    for png in spec.overview(df, out):
        display(Image(filename=str(png)))


def show_processing(spec, bronze_df):
    """Run the Bronze -> Silver transformation and display what it did per feature."""
    silver_df = spec.to_silver(bronze_df)
    display(Markdown(processing_summary_markdown(bronze_df, silver_df, spec.processing)))
    return silver_df


def show_global(dfs, layer):
    """Render the layer-wide global overview, or a placeholder until it exists."""
    out = Path(tempfile.mkdtemp(prefix='nb_global_'))
    imgs = global_overview(dfs, out, layer)
    if not imgs:
        display(Markdown(f'_Global overview {layer}: a venir._'))
        return
    for png in imgs:
        display(Image(filename=str(png)))

## 1. Bronze (raw)

Raw data per source (operators pseudonymised): per-feature understanding, whole-source overview, optional inline analysis; then a layer-wide global overview.

### 1.1 Incidents - Bronze

#### 1.1.1 Incidents - Per feature

In [ ]:
spec = SPECS['incidents']
bronze['incidents'] = spec.load_bronze()
print('incidents bronze', bronze['incidents'].shape)
show_per_feature_spec(spec, bronze['incidents'], spec.bronze_numeric)

#### 1.1.2 Incidents - Overview

In [ ]:
show_overview(SPECS['incidents'], bronze['incidents'])

#### 1.1.3 Incidents - Inline analysis

Incident volume over time, one curve per machine (weekly); then for MACH-03 and MACH-13.

In [ ]:
# Inline (notebook-only): number of incidents over time, one curve per machine.
_idf = bronze['incidents'].copy()
_idf['date'] = pd.to_datetime(_idf['date'], errors='coerce')
_counts = (
    _idf.dropna(subset=['date'])
    .groupby([pd.Grouper(key='date', freq='W'), 'machine_id'])
    .size()
    .unstack('machine_id')
    .fillna(0)
)
fig, ax = plt.subplots(figsize=(13, 6))
for mc in sorted(_counts.columns):
    ax.plot(_counts.index, _counts[mc], linewidth=0.9, label=str(mc))
ax.set_title('Incidents over time by machine (weekly count)')
ax.set_xlabel('date')
ax.set_ylabel('number of incidents')
ax.grid(True, alpha=0.3)
ax.legend(title='machine_id', fontsize=6, ncol=2, loc='upper left', bbox_to_anchor=(1.01, 1.0))
plt.tight_layout()
plt.show()

In [ ]:
# Inline (notebook-only): weekly incident count for two specific machines.
_idf = bronze['incidents'].copy()
_idf['date'] = pd.to_datetime(_idf['date'], errors='coerce')
_counts = (
    _idf.dropna(subset=['date'])
    .groupby([pd.Grouper(key='date', freq='W'), 'machine_id'])
    .size()
    .unstack('machine_id')
    .fillna(0)
)
for _mc in ['MACH-03', 'MACH-13']:
    if _mc not in _counts.columns:
        print(f'{_mc}: no data')
        continue
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(_counts.index, _counts[_mc], color='#4C72B0', linewidth=1.2)
    ax.set_title(f'Incidents over time - {_mc} (weekly count)')
    ax.set_xlabel('date')
    ax.set_ylabel('number of incidents')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### 1.2 Telemetry - Bronze

#### 1.2.1 Telemetry - Per feature

In [ ]:
spec = SPECS['telemetry']
bronze['telemetry'] = spec.load_bronze()
print('telemetry bronze', bronze['telemetry'].shape)
show_per_feature_spec(spec, bronze['telemetry'], spec.bronze_numeric)

#### 1.2.2 Telemetry - Overview

In [ ]:
show_overview(SPECS['telemetry'], bronze['telemetry'])

#### 1.2.3 Telemetry - Inline analysis

One-week zoom on the weekly cyclicity (4 measures + production): mean across machines, then by machine, then for MACH-03 and MACH-13.

In [ ]:
# Inline (notebook-only): one-week zoom to inspect the weekly cyclicity.
_tdf = bronze['telemetry'].copy()
_tdf['t'] = pd.to_datetime(_tdf['timestamp'], errors='coerce')
_start = _tdf['t'].min().normalize()
_week = _tdf[(_tdf['t'] >= _start) & (_tdf['t'] < _start + pd.Timedelta(days=7))]
_measures = ['temperature_c', 'pressure_bar', 'voltage_mean_v', 'rotation_mean_rpm']

# 1) the 4 physical measures over the week (hourly mean across machines)
_hourly = _week.set_index('t')[_measures].resample('h').mean()
fig, axes = plt.subplots(len(_measures), 1, figsize=(12, 8), sharex=True)
for ax, m in zip(axes, _measures):
    ax.plot(_hourly.index, _hourly[m], color='#4C72B0', linewidth=1)
    ax.set_ylabel(m)
    ax.grid(True, alpha=0.3)
axes[0].set_title(f'Telemetry measures - 1 week zoom ({_start.date()}, hourly mean, all machines)')
axes[-1].set_xlabel('timestamp')
plt.tight_layout()
plt.show()

# 2) piece production over the same week (hourly mean across machines)
_prod = _week.set_index('t')['pieces_produced'].resample('h').mean()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(_prod.index, _prod, color='#C44E52', linewidth=1)
ax.set_title(f'Piece production - 1 week zoom ({_start.date()}, hourly mean, all machines)')
ax.set_xlabel('timestamp')
ax.set_ylabel('pieces_produced')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Inline (notebook-only): same one-week zoom, but one curve per machine.
_tdf = bronze['telemetry'].copy()
_tdf['t'] = pd.to_datetime(_tdf['timestamp'], errors='coerce')
_start = _tdf['t'].min().normalize()
_week = _tdf[(_tdf['t'] >= _start) & (_tdf['t'] < _start + pd.Timedelta(days=7))]
_measures = ['temperature_c', 'pressure_bar', 'voltage_mean_v', 'rotation_mean_rpm']
_machines = sorted(_week['machine_id'].dropna().unique())

# 1) the 4 measures over the week, one line per machine
fig, axes = plt.subplots(len(_measures), 1, figsize=(13, 9), sharex=True)
for ax, m in zip(axes, _measures):
    piv = _week.pivot_table(index='t', columns='machine_id', values=m, aggfunc='mean')
    for mc in _machines:
        if mc in piv.columns:
            ax.plot(piv.index, piv[mc], linewidth=0.7, label=str(mc))
    ax.set_ylabel(m)
    ax.grid(True, alpha=0.3)
axes[0].set_title(f'Telemetry measures - 1 week zoom, by machine ({_start.date()})')
axes[-1].set_xlabel('timestamp')
axes[0].legend(title='machine_id', fontsize=6, ncol=2, loc='upper left', bbox_to_anchor=(1.01, 1.0))
plt.tight_layout()
plt.show()

# 2) piece production over the same week, one line per machine
piv = _week.pivot_table(index='t', columns='machine_id', values='pieces_produced', aggfunc='mean')
fig, ax = plt.subplots(figsize=(13, 5))
for mc in _machines:
    if mc in piv.columns:
        ax.plot(piv.index, piv[mc], linewidth=0.7, label=str(mc))
ax.set_title(f'Piece production - 1 week zoom, by machine ({_start.date()})')
ax.set_xlabel('timestamp')
ax.set_ylabel('pieces_produced')
ax.grid(True, alpha=0.3)
ax.legend(title='machine_id', fontsize=6, ncol=2, loc='upper left', bbox_to_anchor=(1.01, 1.0))
plt.tight_layout()
plt.show()

In [ ]:
# Inline (notebook-only): "measures - 1 week zoom" for two specific machines.
_tdf = bronze['telemetry'].copy()
_tdf['t'] = pd.to_datetime(_tdf['timestamp'], errors='coerce')
_start = _tdf['t'].min().normalize()
_week = _tdf[(_tdf['t'] >= _start) & (_tdf['t'] < _start + pd.Timedelta(days=7))]
_measures = ['temperature_c', 'pressure_bar', 'voltage_mean_v', 'rotation_mean_rpm']
for _mc in ['MACH-03', 'MACH-13']:
    _w = _week[_week['machine_id'] == _mc]
    if _w.empty:
        print(f'{_mc}: no data in window')
        continue
    _hourly = _w.set_index('t')[_measures].resample('h').mean()
    fig, axes = plt.subplots(len(_measures), 1, figsize=(12, 8), sharex=True)
    for ax, m in zip(axes, _measures):
        ax.plot(_hourly.index, _hourly[m], color='#4C72B0', linewidth=1)
        ax.set_ylabel(m)
        ax.grid(True, alpha=0.3)
    axes[0].set_title(f'Telemetry measures - 1 week zoom - {_mc} ({_start.date()})')
    axes[-1].set_xlabel('timestamp')
    plt.tight_layout()
    plt.show()

### 1.3 Machines/machine - Bronze

**Browse the `machine` table** (notebook-only): run the cell, then open the **Variables**
view and click the grid icon next to `machine` to sort / filter every row.

In [ ]:
# Inline (notebook-only): browse the raw `machine` table (Variables pane -> Data Viewer).
from src.sources.machines.loader import build_engine, load_machine_referential

pd.set_option('display.max_columns', None)
machine = load_machine_referential(build_engine())   # referential: one row per machine (15)
print('machine', machine.shape)
display(machine)

#### 1.3.1 Machines/machine - Per feature

In [ ]:
spec = SPECS['machine']
bronze['machine'] = spec.load_bronze()
print('machine bronze', bronze['machine'].shape)
show_per_feature_spec(spec, bronze['machine'], spec.bronze_numeric)

#### 1.3.2 Machines/machine - Overview

In [ ]:
from src.sources.machines.overview import machine_tree_markdown

# Overview as a tree: location -> production_line -> criticality -> model -> machine_id
display(Markdown(machine_tree_markdown(bronze['machine'])))

#### 1.3.3 Machines/machine - Inline analysis

_(inline analysis to come)_

### 1.4 Machines/maintenance - Bronze

**Browse the `maintenance` table** (notebook-only): run the cell, then open the
**Variables** view and click the grid icon next to `maintenance` to sort / filter rows.

In [ ]:
# Inline (notebook-only): browse the raw `maintenance` table (Variables pane -> Data Viewer).
from src.sources.machines.loader import build_engine, load_maintenance

pd.set_option('display.max_columns', None)
maintenance = load_maintenance(build_engine())       # maintenance events (1562)
print('maintenance', maintenance.shape)
with pd.option_context('display.max_rows', None):
    display(maintenance)

#### 1.4.1 Machines/maintenance - Per feature

In [ ]:
spec = SPECS['machines']
bronze['machines'] = spec.load_bronze()
print('machines bronze', bronze['machines'].shape)
show_per_feature_spec(spec, bronze['machines'], spec.bronze_numeric)

#### 1.4.2 Machines/maintenance - Overview

In [ ]:
show_overview(SPECS['machines'], bronze['machines'])

#### 1.4.3 Machines/maintenance - Inline analysis

_(inline analysis to come)_

### 1.5 Bronze - Database upload

In [ ]:
from src.database.engine import get_engine, is_available
from src.database.loader import write_table

if is_available():
    engine = get_engine()
    for _name, _df in bronze.items():
        _table = SPECS[_name].table
        _rows = write_table(_df, _table, engine, config.BRONZE_SCHEMA)
        _label = config.SOURCE_DISPLAY_NAMES.get(_name, _name)
        print(f'bronze.{_table:12} <- {_label}: {_rows} rows')
else:
    print('PostgreSQL unavailable - start it with: docker compose up -d')

### 1.6 Bronze - Global overview

Layer-wide views of the Bronze data **read back from the database**: coherence
cross-checks (1.6.1), then cross-source analysis per machine (1.6.2). Requires section 1.5
(Database upload) to have run.

#### 1.6.1 - Coherence cross-check

Cross-source consistency checks on the Bronze data. Each check is shown with an OK/NOK
status (explanations provided when NOK).

In [ ]:
from src.common.coherence import cross_checks_markdown

# Cross-source coherence checks (each shown as ##### <name> (OK/NOK) + explanations).
display(Markdown(cross_checks_markdown(bronze)))

In [ ]:
from src.common.coherence import plot_pieces_vs_capacity

# Illustrate "pieces_produced within machine capacity": one panel per machine, real cumulative
# production (telemetry) vs the theoretical maximum from the daily capacity (machines/machine).
_out = Path(tempfile.mkdtemp(prefix='nb_capacity_'))
for _png in plot_pieces_vs_capacity(bronze, _out):
    display(Image(filename=str(_png)))

#### 1.6.2 - Cross-source analysis

Per-machine cross-source views built from the database (one point per machine, coloured by
criticality).

In [ ]:
# Cross-source analysis from the database (requires 1.5 Bronze - Database upload).
from src.database.engine import get_engine, is_available

if not is_available():
    print('PostgreSQL unavailable - run 1.5 Bronze - Database upload first (docker compose up -d).')
else:
    _eng = get_engine()
    _have = set(pd.read_sql(
        "SELECT table_name FROM information_schema.tables WHERE table_schema = 'bronze'", _eng
    )['table_name'])
    if not {'incidents', 'telemetry', 'machine', 'maintenance'} <= _have:
        print('Bronze tables missing - run 1.5 Bronze - Database upload first.')
    else:
        _inc = pd.read_sql('SELECT machine_id, COUNT(*) AS n_incidents FROM bronze.incidents GROUP BY machine_id', _eng)
        _mnt = pd.read_sql('SELECT machine_id, COUNT(*) AS n_maintenance FROM bronze.maintenance GROUP BY machine_id', _eng)
        _tel = pd.read_sql('SELECT machine_id, AVG(temperature_c) AS mean_temperature_c FROM bronze.telemetry GROUP BY machine_id', _eng)
        _mac = pd.read_sql('SELECT machine_id, criticality FROM bronze.machine', _eng)
        _prof = (_mac.merge(_inc, on='machine_id', how='left')
                     .merge(_mnt, on='machine_id', how='left')
                     .merge(_tel, on='machine_id', how='left'))
        _prof[['n_incidents', 'n_maintenance']] = _prof[['n_incidents', 'n_maintenance']].fillna(0)
        _colors = {'LOW': '#55A868', 'MEDIUM': '#DD8452', 'HIGH': '#C44E52'}

        def _scatter(xc, yc, xl, yl, title):
            display(Markdown(f'##### {title}'))
            fig, ax = plt.subplots(figsize=(9, 6))
            for crit, g in _prof.groupby('criticality'):
                ax.scatter(g[xc], g[yc], s=90, c=_colors.get(crit, '#888'),
                           label=str(crit), edgecolor='black', zorder=3)
            for _, r in _prof.iterrows():
                ax.annotate(str(r['machine_id']).replace('MACH-', ''), (r[xc], r[yc]),
                            fontsize=7, xytext=(3, 3), textcoords='offset points')
            ax.set_xlabel(xl)
            ax.set_ylabel(yl)
            ax.set_title(title)
            ax.grid(True, alpha=0.3)
            ax.legend(title='criticality', fontsize=7)
            plt.tight_layout()
            plt.show()

        _scatter('n_incidents', 'n_maintenance', 'number of incidents', 'number of maintenances',
                 'Incidents vs maintenances per machine')
        _scatter('n_incidents', 'mean_temperature_c', 'number of incidents', 'mean temperature (deg C)',
                 'Mean temperature vs incidents per machine')

In [ ]:
# MACH-03 over one month: telemetry measures + incidents + reactive maintenances overlaid.
from matplotlib.lines import Line2D

_machine = 'MACH-03'
_measures = ['temperature_c', 'pressure_bar', 'voltage_mean_v', 'rotation_mean_rpm']
display(Markdown(
    f'##### {_machine} - telemetry measures with incidents and reactive maintenances (1 month)'
))

# Reactive maintenances for this machine (defines the 1-month window) - maintenance_at is UTC.
_mr = bronze['machines'].query("machine_id == @_machine and maintenance_type == 'reactive'").copy()
_mr['t'] = pd.to_datetime(_mr['maintenance_at'], errors='coerce').dt.tz_localize(None)
_mr = _mr.dropna(subset=['t']).sort_values('t')
_start = _mr['t'].iloc[0].normalize().replace(day=1)        # first day of the first reactive month
_end = _start + pd.DateOffset(months=1)

# Telemetry (hourly measures), incidents (event dates), reactive maintenances - within the window.
_tw = bronze['telemetry'].query("machine_id == @_machine").copy()
_tw['t'] = pd.to_datetime(_tw['timestamp'], errors='coerce')
_tw = _tw[(_tw['t'] >= _start) & (_tw['t'] < _end)].sort_values('t')

_iw = bronze['incidents'].query("machine_id == @_machine").copy()
_iw['t'] = pd.to_datetime(_iw['date'], errors='coerce')
_iw = _iw[(_iw['t'] >= _start) & (_iw['t'] < _end)]

_mw = _mr[(_mr['t'] >= _start) & (_mr['t'] < _end)]

fig, axes = plt.subplots(len(_measures), 1, figsize=(13, 10), sharex=True)
for ax, m in zip(axes, _measures):
    ax.plot(_tw['t'], _tw[m], color='#4C72B0', linewidth=1)
    for x in _iw['t']:
        ax.axvline(x, color='#C44E52', linestyle='--', alpha=0.6, linewidth=1)
    for x in _mw['t']:
        ax.axvline(x, color='black', linestyle=':', alpha=0.8, linewidth=1.2)
    ax.set_ylabel(m)
    ax.grid(True, alpha=0.3)
axes[0].set_title(f'{_machine} - 1 month ({_start.date()}): telemetry measures + incidents + reactive maintenances')
axes[0].legend(handles=[
    Line2D([0], [0], color='#4C72B0', label='telemetry measure'),
    Line2D([0], [0], color='#C44E52', ls='--', label='incident'),
    Line2D([0], [0], color='black', ls=':', label='reactive maintenance'),
], loc='upper left', bbox_to_anchor=(1.01, 1.0), fontsize=8)
axes[-1].set_xlabel('time')
print(f'{_machine} {_start.date()}..{_end.date()} | telemetry {len(_tw)} | incidents {len(_iw)} | reactive {len(_mw)}')
plt.tight_layout()
plt.show()

### 1.7 Bronze - Synthesis

Recap of every feature per source: **status** (OK / NOK) and **suggested processing** to
reach Silver — remediation for each failing criterion (missing → imputation, duplicate →
deduplication, …) plus outlier hypotheses for continuous numeric features.

In [ ]:
from src.common.synthesis import bronze_synthesis_markdown

# Recap table: every feature per source, status + suggested Bronze -> Silver processing.
display(Markdown(bronze_synthesis_markdown(bronze)))

## 2. Processing Bronze -> Silver

Run `to_silver` per source and display, per feature, what it applies (text->value encoding, imputation, IQR outlier clipping, derived features).

### 2.1 Incidents - Processing

In [ ]:
silver['incidents'] = show_processing(SPECS['incidents'], bronze['incidents'])
print('incidents silver', silver['incidents'].shape)

### 2.2 Telemetry - Processing

In [ ]:
silver['telemetry'] = show_processing(SPECS['telemetry'], bronze['telemetry'])
print('telemetry silver', silver['telemetry'].shape)

### 2.3 Machines/machine - Processing

In [ ]:
silver['machine'] = show_processing(SPECS['machine'], bronze['machine'])
print('machine silver', silver['machine'].shape)

### 2.4 Machines/maintenance - Processing

In [ ]:
silver['machines'] = show_processing(SPECS['machines'], bronze['machines'])
print('machines silver', silver['machines'].shape)

## 3. Silver (treated)

Per-feature understanding of the treated data and a whole-source overview; then a layer-wide global overview.

### 3.1 Incidents - Silver

#### 3.1.1 Incidents - Per feature

In [ ]:
spec = SPECS['incidents']
show_per_feature_spec(spec, silver['incidents'], spec.silver_numeric)

#### 3.1.2 Incidents - Overview

In [ ]:
show_overview(SPECS['incidents'], silver['incidents'])

### 3.2 Telemetry - Silver

#### 3.2.1 Telemetry - Per feature

In [ ]:
spec = SPECS['telemetry']
show_per_feature_spec(spec, silver['telemetry'], spec.silver_numeric)

#### 3.2.2 Telemetry - Overview

In [ ]:
show_overview(SPECS['telemetry'], silver['telemetry'])

### 3.3 Machines/machine - Silver

#### 3.3.1 Machines/machine - Per feature

In [ ]:
spec = SPECS['machine']
show_per_feature_spec(spec, silver['machine'], spec.silver_numeric)

#### 3.3.2 Machines/machine - Overview

In [ ]:
from src.sources.machines.overview import machine_tree_markdown

# Overview as a tree: location -> production_line -> criticality -> model -> machine_id
display(Markdown(machine_tree_markdown(silver['machine'])))

### 3.4 Machines/maintenance - Silver

#### 3.4.1 Machines/maintenance - Per feature

In [ ]:
spec = SPECS['machines']
show_per_feature_spec(spec, silver['machines'], spec.silver_numeric)

#### 3.4.2 Machines/maintenance - Overview

In [ ]:
show_overview(SPECS['machines'], silver['machines'])

### 3.5 Silver - Global overview

In [ ]:
show_global(silver, 'silver')

---
Reminder: this notebook is for exploration. The reproducible pipeline is `scripts/run_pipeline.py` (re-run after any data update in `data/raw/`).